In [2]:
aws_sales=pd.read_excel("AWS_Sales.xlsx",
    sheet_name="aws_sales")

customers = pd.read_excel(
    "AWS_Sales.xlsx",
    sheet_name="customers"
)
calendar = pd.read_excel(
    "AWS_Sales.xlsx",
    sheet_name="calendar"
)
territories = pd.read_excel(
    "AWS_Sales.xlsx",
    sheet_name="territories"
)
products_cat = pd.read_excel(
    "AWS_Sales.xlsx",
    sheet_name="products_cat"
)
products_subcat = pd.read_excel(
    "AWS_Sales.xlsx",
    sheet_name="products_subcat"
)
products = pd.read_excel(
    "AWS_Sales.xlsx",
    sheet_name="products"
)
returns = pd.read_excel(
    "AWS_Sales.xlsx",
    sheet_name="returns"
)
product_master = (
    products
    .merge(
        products_subcat,
        on='ProductSubcategoryKey',
        how='left'
    )
    .merge(
        products_cat,
        on='ProductCategoryKey',
        how='left'
    )
)
sales_master=(
    aws_sales.merge(customers,on='CustomerKey',how='left')
    .merge(product_master,on='ProductKey',how='left')
    .merge(territories,left_on='TerritoryKey',right_on='SalesTerritoryKey',how='left')
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
#Product Sales Quantity
product_sales=(sales_master.groupby(['ProductKey','ProductName'])['OrderQuantity'].sum().reset_index())
product_sales

,ProductKey,ProductName,OrderQuantity
0,214,"Sport-100 Helmet, Red",2099
1,215,"Sport-100 Helmet, Black",1940
2,220,"Sport-100 Helmet, Blue",1995
3,223,AWC Logo Cap,4151
4,226,"Long-Sleeve Logo Jersey, S",392
...,...,...,...
125,599,"Mountain-500 Black, 48",55
126,600,"Mountain-500 Black, 52",41
127,604,"Road-750 Black, 44",356
128,605,"Road-750 Black, 48",355


In [6]:
#Product Returns 
product_returns=(returns.groupby('ProductKey')['ReturnQuantity'].sum().reset_index())
product_returns

,ProductKey,ReturnQuantity
0,214,70
1,215,52
2,220,66
3,223,46
4,226,12
...,...,...
119,599,1
120,600,3
121,604,7
122,605,15


In [7]:
#Return analysis
return_analysis=(product_sales.merge(product_returns,on='ProductKey',how='left'))

In [8]:
return_analysis['ReturnQuantity']=(return_analysis['ReturnQuantity'].fillna(0))

In [9]:
#Return Rate
return_analysis['ReturnRate']=(return_analysis['ReturnQuantity']/return_analysis['OrderQuantity'])*100

In [11]:
#Highest Returns Rates
return_analysis.sort_values('ReturnRate',ascending=False)[['ProductName','OrderQuantity','ReturnQuantity','ReturnRate']].head(15)

,ProductName,OrderQuantity,ReturnQuantity,ReturnRate
18,"Road-650 Red, 52",51,6.0,11.764706
27,"Mountain-100 Silver, 44",24,2.0,8.333333
103,"Touring-2000 Blue, 46",96,8.0,8.333333
126,"Mountain-500 Black, 52",41,3.0,7.317073
31,"Mountain-100 Black, 44",31,2.0,6.451613
91,"Touring-3000 Blue, 54",54,3.0,5.555556
32,"Mountain-100 Black, 48",36,2.0,5.555556
17,"Road-650 Red, 48",75,4.0,5.333333
119,"Mountain-500 Silver, 44",38,2.0,5.263158
14,"Road-650 Red, 60",39,2.0,5.128205


Insight
So, high returns were largely driven by high sales volume.
Now the products needing investigation are:

Road-650 Series
Mountain-100 Series
Mountain-500 Series
Touring Series

This is much more concerning because:

Bikes are the primary revenue driver.
Bike returns are expensive.
Bike returns hurt profitability.

In [12]:
#Filter out products with low sales volume
return_analysis[return_analysis['OrderQuantity']>=100].sort_values('ReturnRate',ascending=False)

,ProductKey,ProductName,OrderQuantity,ReturnQuantity,ReturnRate
55,471,"Classic Vest, S",157,8.0,5.095541
60,476,"Women's Mountain Shorts, L",334,17.0,5.089820
9,311,"Road-150 Red, 44",139,7.0,5.035971
88,562,"Touring-1000 Yellow, 50",146,7.0,4.794521
10,312,"Road-150 Red, 48",179,8.0,4.469274
...,...,...,...,...,...
69,485,Fender Set - Mountain,3960,54.0,1.363636
80,536,ML Mountain Tire,2119,28.0,1.321378
75,491,"Short-Sleeve Classic Jersey, XL",383,5.0,1.305483
85,541,Touring Tire,1723,21.0,1.218804


Classic Vest,Women's Mountain Shorts appear near the top.

This suggests:
Possible sizing issues
Customer expectation mismatch
Product quality concerns.

Road-150 Series,Touring-1000 Series also show elevated return rates.

These deserve further investigation because:
Bikes drive ~95% of revenue.
Even small return-rate improvements can significantly impact profit.

Return-count analysis initially identified accessories as the most returned products. However, after normalizing returns by sales volume, apparel products and specific bikes models exhibited the highest return rates. This demonstrated the importance of evaluating performance metrics relative to sales volume rather than relying on absolute counts alone.